# main.tex TTN-TDVP Reproduction

This notebook reproduces the TTN portions of the `main.tex` benchmark workflow through `simulate_tn(..., backend="ttn")` or the GPU-capable `backend="gputtn"`. The bundled `ttn` dispatches to vendored PyTreeNet one-site TDVP on CPU; `gputtn` dispatches to the Julia ITensorNetworks.jl TTN-TDVP bridge and moves the TTN tensors to CUDA storage when `use_cuda=True`.

It covers the paper protocols behind Figs. 4-7: Anneal I, Anneal II, and post-quench dynamics at `h_x/J = 0.5, 2.0, 3.0`. The default cell values run a small smoke test. Set `PAPER_RUN = True` to use the L=10, `V_nn=4`, nearest-neighbor TFIM settings used for the main-text cross-benchmarks. The PyTreeNet backend is CPU-only, while `gputtn` requires the Julia ITensorNetworks project and a functional CUDA/JAX-independent NVIDIA stack.

In [ ]:
from pathlib import Path
import json
import sys
import time

import matplotlib.pyplot as plt
import numpy as np

repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from ryd_gate.analysis.spin_observables import center_line_sites, center_reference_site, line_pairs_from_reference
from ryd_gate.protocols.lattice_dynamics import TFIMAnnealProtocol, TFIMQuenchProtocol
from ryd_gate.backends.tn_common import create_tn_lattice_spec, simulate_tn

## Run Configuration

`PAPER_RUN=True` sets the main-text L=10 benchmark grid and `dt J=0.01`. For a first check, keep it false and use the smoke settings. Increase `CHI_MAX` for convergence studies. `ttn` uses fixed-bond one-site PyTreeNet TDVP; `gputtn` uses ITensorNetworks.jl TDVP with `tdvp_nsites=2` by default.

In [ ]:
PAPER_RUN = False
RUN_SIMULATIONS = True
LOAD_EXISTING = True

if PAPER_RUN:
    L = 10
    CHI_MAX = 64
    DT = 0.01
    T_EVAL_INTERVAL = 0.1
    INITIAL_NOISE = 1e-12
else:
    L = 2
    CHI_MAX = 2
    DT = 0.2
    T_EVAL_INTERVAL = 0.2
    INITIAL_NOISE = 1e-12

BACKEND = "ttn"  # Set to "gputtn" for ITensorNetworks.jl GPU TTN-TDVP.
V_NN = 4.0  # J = V_nn / 4, so V_nn=4 fixes J=1.
INTERACTION_MODE = "nn"  # main.tex TFIM uses nearest-neighbor Ising couplings.
OBSERVABLES = ["sigma_z", "czz_centerline"]
OUTPUT_DIR = repo_root / "scripts" / "notebooks" / "results" / "ttn_main_reproduction"

if BACKEND == "gputtn":
    BACKEND_OPTIONS = {
        "chi_max": CHI_MAX,
        "dt": DT,
        "svd_min": 1e-10,
        "tdvp_nsites": 2,
        "rk_order": 4,
        "use_cuda": True,
        "require_gpu": True,
        "timeout": 600,
    }
else:
    BACKEND_OPTIONS = {
        "chi_max": CHI_MAX,
        "dt": DT,
        "tdvp_order": 1,
        "initial_noise": INITIAL_NOISE,
        "seed": 0,
        "progress": False,
    }

print(json.dumps({
    "paper_run": PAPER_RUN,
    "L": L,
    "backend": BACKEND,
    "backend_options": BACKEND_OPTIONS,
    "interaction_mode": INTERACTION_MODE,
    "output_dir": str(OUTPUT_DIR),
}, indent=2))

## Paper Protocols

The Hamiltonian units are chosen so that `J = 1`. Anneal I enters the AFM dome by changing `h_x(t)` with `h_z/J = 0`; Anneal II enters from the side with `h_x^m/J = 0.5` and a longitudinal-field sweep. The quench protocols use `h_z/J = 0` and `h_x/J in {0.5, 2.0, 3.0}`.

In [ ]:
ANNEAL_CASES = {
    "anneal_I": {
        "kind": "anneal",
        "protocol": TFIMAnnealProtocol(
            hx_peak=3.5,
            hx_initial=0.0,
            hx_final=0.0,
            hz_initial=0.0,
            hz_final=0.0,
            t_rise=1.5,
            t_sweep=1.5,
            t_fall=3.0,
        ),
        "t_profile": [3.0, 6.0],
        "label": "Anneal I",
    },
    "anneal_II": {
        "kind": "anneal",
        "protocol": TFIMAnnealProtocol(
            hx_peak=0.5,
            hx_initial=0.0,
            hx_final=0.0,
            hz_initial=-8.0,
            hz_final=0.0,
            t_rise=1.5,
            t_sweep=3.0,
            t_fall=1.5,
        ),
        "t_profile": [3.8, 6.0],
        "label": "Anneal II",
    },
}

QUENCH_CASES = {
    f"quench_hx_{str(hx).replace('.', 'p')}": {
        "kind": "quench",
        "protocol": TFIMQuenchProtocol(hx=hx, hz=0.0, t_gate=3.0),
        "hx": hx,
        "t_profile": [1.5],
        "label": f"Quench h_x/J={hx}",
    }
    for hx in [0.5, 2.0, 3.0]
}

CASES = {**ANNEAL_CASES, **QUENCH_CASES}
for name, case in CASES.items():
    print(name, case["label"], "t_gate=", case["protocol"].t_gate)

## Simulation Helpers

Each run saves an `.npz` with recorded times, observables, and metadata. Re-running the notebook can load existing data instead of recomputing.

In [ ]:
def make_spec():
    return create_tn_lattice_spec(
        L,
        L,
        V_nn=V_NN,
        Omega=1.0,
        level_structure="1r",
        interaction_mode=INTERACTION_MODE,
        ordering="snake",
    )


def t_eval_for(protocol):
    return np.arange(0.0, protocol.t_gate + 1e-12, T_EVAL_INTERVAL)


def result_path(case_name):
    mode = "paper" if PAPER_RUN else "smoke"
    return OUTPUT_DIR / f"{case_name}_L{L}_chi{CHI_MAX}_dt{str(DT).replace('.', 'p')}_{mode}_{BACKEND}.npz"


def save_result(case_name, case, result, elapsed_s):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    path = result_path(case_name)
    arrays = {"times": np.asarray(result.times, dtype=float)}
    for obs_name, value in (result.metadata.get("obs") or {}).items():
        arrays[f"obs_{obs_name}"] = np.asarray(value)
    metadata = {
        "case": case_name,
        "label": case["label"],
        "kind": case["kind"],
        "L": L,
        "V_nn": V_NN,
        "J": V_NN / 4.0,
        "interaction_mode": INTERACTION_MODE,
        "backend": BACKEND,
        "backend_options": BACKEND_OPTIONS,
        "result_backend": result.metadata.get("backend"),
        "result_method": result.metadata.get("method"),
        "result_tree": result.metadata.get("tree"),
        "result_gpu": result.metadata.get("gpu"),
        "elapsed_s": elapsed_s,
    }
    if "hx" in case:
        metadata["hx"] = case["hx"]
    arrays["metadata_json"] = np.asarray(json.dumps(metadata, sort_keys=True))
    np.savez(path, **arrays)
    return path


def load_result(case_name):
    path = result_path(case_name)
    data = np.load(path, allow_pickle=False)
    obs = {
        key.removeprefix("obs_"): np.asarray(data[key])
        for key in data.files
        if key.startswith("obs_")
    }
    metadata = json.loads(str(np.asarray(data["metadata_json"]).item()))
    return {"times": np.asarray(data["times"]), "obs": obs, "metadata": metadata, "path": path}


def run_case(case_name, case):
    path = result_path(case_name)
    if LOAD_EXISTING and path.exists():
        print(f"loaded {case_name}: {path}")
        return load_result(case_name)
    if not RUN_SIMULATIONS:
        print(f"skip {case_name}; RUN_SIMULATIONS=False and no cached result at {path}")
        return None

    spec = make_spec()
    protocol = case["protocol"]
    start = time.perf_counter()
    result = simulate_tn(
        spec,
        protocol,
        [],
        initial_state="all_ground",
        backend=BACKEND,
        t_eval=t_eval_for(protocol),
        observables=OBSERVABLES,
        backend_options=BACKEND_OPTIONS,
    )
    elapsed = time.perf_counter() - start
    path = save_result(case_name, case, result, elapsed)
    print(f"saved {case_name}: {path} ({elapsed:.2f}s)")
    return load_result(case_name)

In [ ]:
results = {}
for case_name, case in CASES.items():
    results[case_name] = run_case(case_name, case)

results = {name: value for name, value in results.items() if value is not None}
print("available results:", sorted(results))

## Observable Helpers

`sigma_z` is recorded for every site in row-major order. `czz_centerline` is the connected `sigma_z sigma_z` correlation between the center reference site and every other site on the same horizontal center line.

In [ ]:
center_site = center_reference_site(L, L)
line_sites = center_line_sites(L, L, axis="horizontal")
pairs = line_pairs_from_reference(L, L, axis="horizontal")
pair_distances = np.array([abs(j % L - center_site % L) for _, j in pairs], dtype=int)
nearest_corr_idx = int(np.argmin(pair_distances)) if len(pair_distances) else 0


def nearest_time_index(times, t_target):
    return int(np.argmin(np.abs(np.asarray(times) - float(t_target))))


def center_sigma_trace(result):
    return result["obs"]["sigma_z"][:, center_site]


def nearest_czz_trace(result):
    czz = result["obs"]["czz_centerline"]
    if czz.shape[1] == 0:
        return np.zeros(czz.shape[0])
    return czz[:, nearest_corr_idx]


def line_sigma_profile(result, t_target):
    idx = nearest_time_index(result["times"], t_target)
    return result["obs"]["sigma_z"][idx, line_sites]


def czz_profile(result, t_target):
    idx = nearest_time_index(result["times"], t_target)
    return result["obs"]["czz_centerline"][idx]


print("center_site", center_site, "line_sites", line_sites.tolist(), "nearest_corr_idx", nearest_corr_idx)

## Figs. 4-5 Style: Annealing Time Traces

These panels reproduce the TTN curves corresponding to local center magnetization and nearest-neighbor center-line connected correlation for Anneal I and II.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex="col")
for row, case_name in enumerate(["anneal_I", "anneal_II"]):
    if case_name not in results:
        continue
    result = results[case_name]
    case = CASES[case_name]
    times = result["times"]
    axes[row, 0].plot(times, center_sigma_trace(result), color="tab:blue")
    axes[row, 1].plot(times, nearest_czz_trace(result), color="tab:green")
    for t_marker in case["t_profile"]:
        axes[row, 0].axvline(t_marker, color="0.4", linestyle="--", linewidth=1)
        axes[row, 1].axvline(t_marker, color="0.4", linestyle="--", linewidth=1)
    axes[row, 0].set_title(case["label"])
    axes[row, 0].set_ylabel(r"$\langle \sigma^z_{r_0} \rangle$")
    axes[row, 1].set_ylabel(r"$C(\delta_1)$")
for ax in axes[-1, :]:
    ax.set_xlabel("tJ")
for ax in axes.ravel():
    ax.grid(alpha=0.2)
fig.tight_layout()

## Figs. 4-5 Style: Annealing Center-Line Profiles

The paper compares profiles at a near-crossing time and at the final time. The markers below use `tJ = 3.0, 6.0` for Anneal I and `tJ = 3.8, 6.0` for Anneal II.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
x_line = np.arange(len(line_sites))
for row, case_name in enumerate(["anneal_I", "anneal_II"]):
    if case_name not in results:
        continue
    result = results[case_name]
    case = CASES[case_name]
    for t_marker in case["t_profile"]:
        axes[row, 0].plot(x_line, line_sigma_profile(result, t_marker), marker="o", label=f"tJ={t_marker}")
        axes[row, 1].plot(pair_distances, czz_profile(result, t_marker), marker="o", label=f"tJ={t_marker}")
    axes[row, 0].set_title(f"{case['label']} sigma_z line")
    axes[row, 1].set_title(f"{case['label']} C(delta) line")
    axes[row, 0].set_ylabel(r"$\langle \sigma^z_{r_i} \rangle$")
    axes[row, 1].set_ylabel(r"$C(\delta_i)$")
for ax in axes.ravel():
    ax.set_xlabel("center-line index / distance")
    ax.grid(alpha=0.2)
    ax.legend()
fig.tight_layout()

## Figs. 6-7 Style: Post-Quench Time Traces

These are the TTN curves for the three post-quench values `h_x/J = 0.5, 2.0, 3.0` at `h_z/J = 0`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for case_name, case in QUENCH_CASES.items():
    if case_name not in results:
        continue
    result = results[case_name]
    label = f"h_x/J={case['hx']}"
    axes[0].plot(result["times"], center_sigma_trace(result), label=label)
    axes[1].plot(result["times"], nearest_czz_trace(result), label=label)
axes[0].set_xlabel("tJ")
axes[0].set_ylabel(r"$\langle \sigma^z_{r_0} \rangle$")
axes[1].set_xlabel("tJ")
axes[1].set_ylabel(r"$C(\delta_1)$")
for ax in axes:
    ax.axvline(1.5, color="0.4", linestyle="--", linewidth=1)
    ax.grid(alpha=0.2)
    ax.legend()
fig.tight_layout()

## Figs. 6-7 Style: Post-Quench Profiles at t_Q J = 1.5

The paper uses `t_Q J = 1.5` for the spatial profile comparisons.

In [ ]:
T_Q = 1.5
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
x_line = np.arange(len(line_sites))
for case_name, case in QUENCH_CASES.items():
    if case_name not in results:
        continue
    result = results[case_name]
    label = f"h_x/J={case['hx']}"
    axes[0].plot(x_line, line_sigma_profile(result, T_Q), marker="o", label=label)
    axes[1].plot(pair_distances, czz_profile(result, T_Q), marker="o", label=label)
axes[0].set_xlabel("center-line index")
axes[0].set_ylabel(r"$\langle \sigma^z_{r_i} \rangle$")
axes[1].set_xlabel(r"$|\delta_i|$")
axes[1].set_ylabel(r"$C(\delta_i)$")
for ax in axes:
    ax.grid(alpha=0.2)
    ax.legend()
fig.tight_layout()

## Optional Baseline Comparison

If you have MPS or 2DTN `.npz` outputs with the same observable keys, set `REFERENCE_PATHS` and run this cell to compute the paper-style profile discrepancies at representative times.

In [ ]:
REFERENCE_PATHS = {}  # Example: {"anneal_I": Path("scripts/results/...npz")}


def load_npz_observables(path):
    data = np.load(path, allow_pickle=False)
    obs = {key.removeprefix("obs_"): np.asarray(data[key]) for key in data.files if key.startswith("obs_")}
    return {"times": np.asarray(data["times"]), "obs": obs, "path": Path(path)}


def epsilon_z_profile(profile_a, profile_b):
    return float(2.0 * np.sum(np.abs(np.asarray(profile_a) - np.asarray(profile_b))) / len(profile_a))


def epsilon_zz_profile(profile_a, profile_b, floor=1e-15):
    denom = max(float(np.sum(np.abs(profile_a))), floor)
    return float(np.sum(np.abs(np.asarray(profile_a) - np.asarray(profile_b))) / denom)


comparisons = []
for case_name, ref_path in REFERENCE_PATHS.items():
    if case_name not in results:
        continue
    ref = load_npz_observables(ref_path)
    result = results[case_name]
    for t_marker in CASES[case_name]["t_profile"]:
        comparisons.append({
            "case": case_name,
            "tJ": t_marker,
            "epsilon_z": epsilon_z_profile(line_sigma_profile(ref, t_marker), line_sigma_profile(result, t_marker)),
            "epsilon_zz": epsilon_zz_profile(czz_profile(ref, t_marker), czz_profile(result, t_marker)),
            "reference": str(ref_path),
        })

comparisons